# E2FGVI-HQ CUDA benchmark (FP32)
Giữ nguyên thuật toán/cấu hình CPU thắng. Chạy smoke 3 frame, rồi 48 frame; full video mặc định bị khóa.

In [17]:
import torch
assert torch.cuda.is_available(), 'Hãy chọn Runtime > Change runtime type > GPU rồi reconnect'
gpu = torch.cuda.get_device_properties(0)
print({
    'gpu': gpu.name,
    'cuda_version': torch.version.cuda,
    'pytorch_version': torch.__version__,
    'vram_gb': round(gpu.total_memory / 1024**3, 2),
})

{'gpu': 'Tesla T4', 'cuda_version': '12.8', 'pytorch_version': '2.11.0+cu128', 'vram_gb': 14.56}


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
# Source lấy từ GitHub; Drive chỉ giữ dữ liệu lớn và kết quả.
REPO_URL = 'https://github.com/duchieu34/xoa-logo.git'
BRANCH = 'research/e2fgvi-hq-colab-gpu'
DRIVE_ROOT = '/content/drive/MyDrive/veo'
VIDEO_IN_DRIVE = f'{DRIVE_ROOT}/ft-vid-23.mp4'
CHECKPOINT_IN_DRIVE = f'{DRIVE_ROOT}/E2FGVI-HQ-CVPR22.pth'
PROJECT_DIR = '/content/Xoa-logo-video'
GPU_RESULTS_IN_DRIVE = f'{DRIVE_ROOT}/results/e2fgvi_colab_gpu'
# Tự tìm theo tên nếu file không nằm đúng trong MyDrive/veo.
from pathlib import Path
drive_root = Path('/content/drive/MyDrive')
video_candidates = list(drive_root.rglob('ft-vid-23.mp4'))
checkpoint_candidates = list(drive_root.rglob('E2FGVI-HQ-CVPR22.pth'))
print('Video candidates:', video_candidates)
print('Checkpoint candidates:', checkpoint_candidates)
if video_candidates:
    VIDEO_IN_DRIVE = str(video_candidates[0])
if checkpoint_candidates:
    CHECKPOINT_IN_DRIVE = str(checkpoint_candidates[0])
print('Using video:', VIDEO_IN_DRIVE)
print('Using checkpoint:', CHECKPOINT_IN_DRIVE)

Video candidates: [PosixPath('/content/drive/MyDrive/veo/ft-vid-23.mp4')]
Checkpoint candidates: [PosixPath('/content/drive/MyDrive/veo/E2FGVI-HQ-CVPR22.pth')]
Using video: /content/drive/MyDrive/veo/ft-vid-23.mp4
Using checkpoint: /content/drive/MyDrive/veo/E2FGVI-HQ-CVPR22.pth


In [23]:
import os
import subprocess
from pathlib import Path
for required in (VIDEO_IN_DRIVE, CHECKPOINT_IN_DRIVE):
    assert Path(required).is_file(), f'Missing: {required}'
project = Path(PROJECT_DIR)
if (project / '.git').is_dir():
    subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)
elif project.exists():
    raise RuntimeError(f'{PROJECT_DIR} tồn tại nhưng không phải Git repository; hãy xóa runtime rồi chạy lại')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
print('Project:', Path.cwd())
source = Path('research/e2fgvi_hq/benchmark.py').read_text(encoding='utf-8')
assert '--device' in source, 'Nhánh Git hiện tại chưa có hỗ trợ --device'

Project: /content/Xoa-logo-video


In [33]:
%cd /content/Xoa-logo-video
!git pull origin research/e2fgvi-hq-colab-gpu

/content
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 568 bytes | 284.00 KiB/s, done.
From https://github.com/duchieu34/xoa-logo
 * branch            research/e2fgvi-hq-colab-gpu -> FETCH_HEAD
   de1a9d5..a56933c  research/e2fgvi-hq-colab-gpu -> origin/research/e2fgvi-hq-colab-gpu
Updating de1a9d5..a56933c
Fast-forward
 research/e2fgvi_hq/full_video_validation.py | 10 +++++++++-
 1 file changed, 9 insertions(+), 1 deletion(-)


In [34]:
!grep -n "_report_path" research/e2fgvi_hq/full_video_validation.py

53:def _report_path(path: Path) -> str:
395:            "path": _report_path(output_video),


In [24]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
# Không cài requirements-ai.txt vì file đó cố ý pin bản PyTorch CPU.
!pip install -q 'numpy>=1.26,<3' 'opencv-python-headless>=4.10,<5' psutil==7.0.0 gdown==5.2.0 pytest
!mkdir -p samples research/e2fgvi_hq/checkpoints third_party
!cp "{VIDEO_IN_DRIVE}" samples/ft-vid-23.mp4
!cp "{CHECKPOINT_IN_DRIVE}" research/e2fgvi_hq/checkpoints/E2FGVI-HQ-CVPR22.pth
![ -d third_party/E2FGVI/.git ] || git clone https://github.com/MCG-NKU/E2FGVI.git third_party/E2FGVI
!git -C third_party/E2FGVI checkout 709cbe319edc21b8a365a28e14cba595a93d62cf

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 17.5 MB/s eta 0:00:00:00:0100:01
Cloning into 'third_party/E2FGVI'...
remote: Enumerating objects: 345, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 345 (delta 51), reused 30 (delta 30), pack-reused 265 (from 1)
Receiving objects: 100% (345/345), 36.75 MiB | 30.37 MiB/s, done.
Resolving deltas: 100% (54/54), done.
Note: switching to '709cbe319edc21b8a365a28e14cba595a93d62cf'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you 

In [25]:
# Smoke test thực tế: 3 frame, FP32 CUDA, cùng crop/mask/temporal config.
!python -m research.e2fgvi_hq.benchmark --device cuda --start-frame 130 --frames 3 --crop-size 192 --neighbor-stride 5 --reference-step 10 --aggregation legacy_average --threads 4 --skip-baselines --output-dir /content/e2fgvi_gpu_smoke --report /content/e2fgvi_gpu_smoke_report.json

load pretrained SPyNet...
/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
{
  "experiment": "E2FGVI-HQ CUDA proof of concept",
  "source": "samples/ft-vid-23.mp4",
  "segment": {
    "start_frame": 130,
    "end_frame_inclusive": 132,
    "frame_count": 3,
    "fps": 24.0,
    "duration_seconds": 0.125,
    "contains_transition_130_to_131": true
  },
  "runtime": {
    "device": "cuda",
    "device_name": "Tesla T4",
    "torch": "2.11.0+cu128",
    "torch_cuda_build": "12.8",
    "cuda_available": true,
    "threads": 4,
    "model_load_seconds": 0.865057,
    "inference_total_seconds": 1.656326,
    "inference_seconds_per_output_frame": 0.552109,
    "inference_output_fps": 1.811238,
    "total_runtime_seconds": 12.526

In [27]:
import json
smoke = json.loads(Path('/content/e2fgvi_gpu_smoke_report.json').read_text())
assert smoke['runtime']['device'] == 'cuda'
assert smoke['inference_windows']['outside_mask_max_absolute_change'] == 0
print(smoke['runtime'])
SMOKE_OK = True

{'device': 'cuda', 'device_name': 'Tesla T4', 'torch': '2.11.0+cu128', 'torch_cuda_build': '12.8', 'cuda_available': True, 'threads': 4, 'model_load_seconds': 0.865057, 'inference_total_seconds': 1.656326, 'inference_seconds_per_output_frame': 0.552109, 'inference_output_fps': 1.811238, 'total_runtime_seconds': 12.526467, 'rss_before_model_mb': 769.926, 'rss_after_model_mb': 1033.227, 'peak_rss_mb': 1447.602, 'peak_vram_mb': 310.967, 'total_vram_mb': 14912.688}


In [28]:
# Benchmark chuẩn: frame 108–155 (48 frame). Chỉ chạy sau smoke thành công.
assert SMOKE_OK
!python -m research.e2fgvi_hq.benchmark --device cuda --start-frame 108 --frames 48 --crop-size 192 --neighbor-stride 5 --reference-step 10 --aggregation legacy_average --threads 4 --skip-baselines --output-dir /content/e2fgvi_gpu_48 --report /content/e2fgvi_gpu_48_report.json

load pretrained SPyNet...
/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
{
  "experiment": "E2FGVI-HQ CUDA proof of concept",
  "source": "samples/ft-vid-23.mp4",
  "segment": {
    "start_frame": 108,
    "end_frame_inclusive": 155,
    "frame_count": 48,
    "fps": 24.0,
    "duration_seconds": 2.0,
    "contains_transition_130_to_131": true
  },
  "runtime": {
    "device": "cuda",
    "device_name": "Tesla T4",
    "torch": "2.11.0+cu128",
    "torch_cuda_build": "12.8",
    "cuda_available": true,
    "threads": 4,
    "model_load_seconds": 0.712105,
    "inference_total_seconds": 5.358829,
    "inference_seconds_per_output_frame": 0.111642,
    "inference_output_fps": 8.957181,
    "total_runtime_seconds": 16.0237

In [29]:
import shutil
gpu48 = json.loads(Path('/content/e2fgvi_gpu_48_report.json').read_text())
runtime = gpu48['runtime']
print({
    'gpu_s_per_frame': runtime['inference_seconds_per_output_frame'],
    'gpu_fps': runtime['inference_output_fps'],
    'peak_vram_mb': runtime['peak_vram_mb'],
    'speedup_vs_cpu_48': round(2.487 / runtime['inference_seconds_per_output_frame'], 2),
    'transitions': {k: v for k, v in gpu48['metrics']['e2fgvi_hq'].items() if 'transition_' in k},
})
Path(GPU_RESULTS_IN_DRIVE).mkdir(parents=True, exist_ok=True)
shutil.copy2('/content/e2fgvi_gpu_48_report.json', f'{GPU_RESULTS_IN_DRIVE}/benchmark_48_fp32.json')
shutil.make_archive('/content/e2fgvi_gpu_48_diagnostics', 'zip', '/content/e2fgvi_gpu_48')
shutil.copy2('/content/e2fgvi_gpu_48_diagnostics.zip', GPU_RESULTS_IN_DRIVE)
GPU_48_OK = True

{'gpu_s_per_frame': 0.111642, 'gpu_fps': 8.957181, 'peak_vram_mb': 1668.282, 'speedup_vs_cpu_48': 22.28, 'transitions': {'worst_transition_to_absolute_frame': 133, 'worst_transition_mad': 54.981595, 'transition_130_to_131_mad': 31.242331, 'transition_130_to_131_mean_luma_delta': 8.088959, 'transition_131_to_132_mad': 45.122699, 'transition_131_to_132_mean_luma_delta': 0.380371, 'transition_132_to_133_mad': 54.981595, 'transition_132_to_133_mean_luma_delta': -15.668716, 'transition_133_to_134_mad': 41.423313, 'transition_133_to_134_mean_luma_delta': -19.184048}}


In [ ]:
# Chỉ đổi thành True sau khi đã xem diagnostics 48 frame và chấp nhận chất lượng.
RUN_FULL_VIDEO = True
if RUN_FULL_VIDEO:
    assert GPU_48_OK
    subprocess.run([
        'python', '-m', 'research.e2fgvi_hq.full_video_validation',
        '--device', 'cuda', '--video', 'samples/ft-vid-23.mp4',
        '--checkpoint', 'research/e2fgvi_hq/checkpoints/E2FGVI-HQ-CVPR22.pth',
        '--crop-size', '192', '--neighbor-stride', '5', '--reference-step', '10',
        '--aggregation', 'legacy_average', '--threads', '4',
        '--output-dir', '/content/e2fgvi_gpu_full',
        '--report', '/content/e2fgvi_gpu_full_report.json',
    ], check=True)
    shutil.copy2('/content/e2fgvi_gpu_full_report.json', f'{GPU_RESULTS_IN_DRIVE}/full_192_fp32.json')
    shutil.copy2('/content/e2fgvi_gpu_full/ft-vid-23_e2fgvi_hq_cuda_full.mp4', GPU_RESULTS_IN_DRIVE)
else:
    print('Full 192-frame run is locked. Review the 48-frame report/diagnostics first.')

In [32]:
!ls -lah /content/e2fgvi_gpu_full
!find /content -maxdepth 2 -iname "*report*.json" -o -iname "*.mp4"

total 25M
drwxr-xr-x 3 root root 4.0K Aug 31 12:06 .
drwxr-xr-x 1 root root 4.0K Aug 31 12:06 ..
-rw-r--r-- 1 root root  25M Aug 31 12:06 ft-vid-23_e2fgvi_hq_cuda_full.mp4
drwxr-xr-x 2 root root 4.0K Aug 31 12:06 top_transitions
/content/e2fgvi_gpu_smoke_report.json
/content/e2fgvi_gpu_smoke/five_method_comparison.mp4
/content/e2fgvi_gpu_smoke/e2fgvi_hq_crop.mp4
/content/e2fgvi_gpu_full/ft-vid-23_e2fgvi_hq_cuda_full.mp4
/content/e2fgvi_gpu_48_report.json
/content/e2fgvi_gpu_48/five_method_comparison.mp4
/content/e2fgvi_gpu_48/e2fgvi_hq_crop.mp4


## FP16
Chưa được triển khai trong notebook này. Chỉ mở experiment FP16 riêng sau khi FP32 CUDA chạy đúng và output tương đương CPU.